# Phase 4b — QLoRA fine-tune of the 8B chess coach (RunPod)

Fine-tunes **Qwen3-8B** on the same `v1` distillation dataset as the 3B Colab run, then
quantizes to GGUF and **terminates its own pod** when done. Artifacts persist on the attached
network volume (`/workspace`).

Qwen3 is a hybrid-reasoning family; we train and deploy it in **non-thinking mode** (plain
coaching answers, no `<think>` blocks) — the sanity-gen cell passes `enable_thinking=False`,
and the Mac-side Ollama client must disable thinking too.

**Verifying on Colab first (free):** the notebook detects where it runs via `$RUNPOD_POD_ID`.
Off RunPod it switches to **smoke-test mode**: `/content` paths, fp16 on bf16-less GPUs (T4),
`max_steps=3` training, an upload prompt for the jsonls, and self-terminate becomes a no-op.
On a T4 the merge/GGUF steps run via slow CPU/disk offload — treat them as best-effort there;
they run cleanly on-GPU on the real pod.

**Pod setup (once, in the RunPod console):**
1. Create a **network volume** (≥ 25 GB) in a data center that has your GPU in stock.
2. Deploy a **secure-cloud GPU pod** in that same DC with the volume attached:
   - GPU: a **24 GB card (RTX A5000) is enough** — the 8B fp16 merge (~16 GB) fits on-GPU.
   - Template: latest RunPod PyTorch
   - Container disk: **100 GB** (ephemeral scratch: HF cache + merged fp16 + f16 GGUF)
   - Volume mount path: `/workspace` (the default)
3. Open Jupyter; upload this notebook to `/workspace/` and `train.jsonl` / `val.jsonl` to
   `/workspace/data/`.
4. **Run All.** Safe to close the browser — cells keep executing server-side.

**Persistence contract:** everything under `/workspace` survives pod termination; everything
else is scratch. The last cell removes the pod (billing stops) only after verifying the final
GGUF landed on the volume. **If any cell errors, Run All stops and the pod keeps billing —
check back, or stop it from the console.**

## 0 · GPU

In [ ]:
import os, torch

ON_RUNPOD = "RUNPOD_POD_ID" in os.environ
SMOKE = not ON_RUNPOD            # off-RunPod (e.g. Colab) = quick mechanics check
BF16 = torch.cuda.is_bf16_supported()   # T4 has no bf16 -> fp16 fallback

assert torch.cuda.is_available(), "No GPU visible"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} ({VRAM_GB:.0f} GB) | runpod={ON_RUNPOD} smoke={SMOKE} bf16={BF16}")
!nvidia-smi --query-gpu=memory.total,memory.used --format=csv

## 1 · Install

In [ ]:
%pip install -q -U transformers peft trl bitsandbytes accelerate datasets
# some images preinstall a torchao too old for newer peft (raises on any model load);
# nothing here uses it, so remove it rather than upgrade.
%pip uninstall -q -y torchao

## 2 · Paths & data
The volume is the persistence boundary: dataset in, adapter + GGUF + manifest out.
The 28 GB base-model download is re-fetchable, so the HF cache stays on scratch.

In [ ]:
import json

VOL = "/workspace" if ON_RUNPOD else "/content"      # network volume on RunPod; scratch on Colab
SCRATCH = "/root/scratch" if ON_RUNPOD else "/content/scratch"
ART, DATA = f"{VOL}/artifacts", f"{VOL}/data"
for d in (ART, DATA, SCRATCH):
    os.makedirs(d, exist_ok=True)
os.environ["HF_HOME"] = f"{SCRATCH}/hf"

if ON_RUNPOD:
    assert os.path.ismount(VOL), "/workspace is not a mounted network volume — artifacts would die with the pod"

missing = [f for f in ("train.jsonl", "val.jsonl") if not os.path.exists(f"{DATA}/{f}")]
if missing and not ON_RUNPOD:
    from google.colab import files
    print("Select", " and ".join(missing), "…")
    for name in files.upload():
        os.replace(name, f"{DATA}/{name}")

for f in ("train.jsonl", "val.jsonl"):
    p = f"{DATA}/{f}"
    assert os.path.exists(p), f"missing {p} — upload it to {DATA}/ first"
    print(f, sum(1 for _ in open(p)), "examples")

## 3 · Base model, 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MODEL = "Qwen/Qwen3-8B"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if BF16 else torch.float16,
)
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16 if BF16 else torch.float16,
)
# make the 4-bit model trainable-through: cast norms to fp32, enable grad checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False          # incompatible with gradient checkpointing
print(model.get_memory_footprint() / 1e9, "GB resident (4-bit base)")

## 4 · LoRA

In [ ]:
from peft import LoraConfig, get_peft_model

lora = LoraConfig(
    r=16,                 # same capacity as the 3B run — the 82-example set is the constraint
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],   # all linear layers
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 5 · Dataset

In [ ]:
# Recent TRL applies the chat template automatically for {"messages":[...]} rows.
from datasets import load_dataset
ds = load_dataset("json", data_files={"train": f"{DATA}/train.jsonl", "validation": f"{DATA}/val.jsonl"})
print(ds)
print("\nexample gold coaching:\n", ds["train"][0]["messages"][-1]["content"][:300])

## 6 · Train

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=f"{SCRATCH}/ckpt-8b",
    num_train_epochs=3,
    max_steps=3 if SMOKE else -1,   # smoke mode: mechanics only
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,     # effective batch = 16, same as the 3B run
    learning_rate=1e-4,                # halved vs the 3B run — bigger model, gentler LoRA lr
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=BF16,
    fp16=not BF16,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    max_length=2048,
    report_to="none",
    seed=0,
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    processing_class=tok,
)
trainer.train()

## 7 · Save adapter → volume
The adapter is the training run's irreplaceable output — persist it before anything fragile.

In [ ]:
ADAPTER = f"{ART}/qwen-coach-8b-lora-v1"
trainer.save_model(ADAPTER)
tok.save_pretrained(ADAPTER)
!du -sh {ADAPTER}

## 8 · Sanity generation

In [ ]:
model.eval()
sample = json.loads(open(f"{DATA}/val.jsonl").readline())["messages"]
prompt = [m for m in sample if m["role"] in ("system", "user")]

inputs = tok.apply_chat_template(
    prompt, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    enable_thinking=False,   # Qwen3: plain answer, no <think> block
).to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False)

gen = out[0][inputs["input_ids"].shape[1]:]
print("FACTS:\n", prompt[-1]["content"][:500])
print("\nCOACH (fine-tuned 8B):\n", tok.decode(gen, skip_special_tokens=True))

## 9 · Merge fp16
Needs ~16 GB on-GPU (fine on the 24 GB pod). peft's disk-offload merge is buggy
(KeyError in `_update_offload`), so sub-20 GB GPUs skip this instead of crashing.

In [ ]:
del model, trainer
torch.cuda.empty_cache()

DO_MERGE = VRAM_GB >= 20
if DO_MERGE:
    from peft import AutoPeftModelForCausalLM
    MERGED = f"{SCRATCH}/qwen-coach-8b-merged"
    merged = AutoPeftModelForCausalLM.from_pretrained(
        ADAPTER, torch_dtype=torch.float16, device_map="auto",
    ).merge_and_unload()
    merged.save_pretrained(MERGED, safe_serialization=True, max_shard_size="5GB")
    tok.save_pretrained(MERGED)

    # newer transformers writes an "extra_special_tokens" list that the older transformers
    # pinned by llama.cpp's requirements can't parse — strip it before conversion reads it
    cfgp = f"{MERGED}/tokenizer_config.json"
    cfg = json.load(open(cfgp))
    if isinstance(cfg.get("extra_special_tokens"), list):
        del cfg["extra_special_tokens"]
        json.dump(cfg, open(cfgp, "w"), indent=2)
    print("merged fp16 model saved")
else:
    print(f"skipping merge on a {VRAM_GB:.0f} GB GPU — needs ~16 GB on-GPU; runs on the pod")

## 10 · GGUF (f16 → q4_k_m) → volume

In [ ]:
GGUF_OUT = f"{ART}/qwen-coach-8b-v1-q4_k_m.gguf"
if DO_MERGE:
    GGUF_F16 = f"{SCRATCH}/qwen-coach-8b-v1-f16.gguf"
    !test -d {SCRATCH}/llama.cpp || git clone --depth 1 https://github.com/ggerganov/llama.cpp {SCRATCH}/llama.cpp
    !pip install -q -r {SCRATCH}/llama.cpp/requirements.txt
    !python {SCRATCH}/llama.cpp/convert_hf_to_gguf.py {MERGED} --outfile {GGUF_F16} --outtype f16
    !cd {SCRATCH}/llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release >/dev/null && cmake --build build -j --target llama-quantize >/dev/null
    !{SCRATCH}/llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_OUT} q4_k_m
    !ls -lh {GGUF_OUT}
else:
    print("skipping GGUF — no merged model (smoke on a small GPU)")

## 11 · Run manifest → volume

In [ ]:
import datetime, subprocess
manifest = {
    "run": "qwen-coach-8b-v1",
    "base_model": MODEL,
    "thinking_mode": False,
    "dataset": "v1 (82 train / 9 val)",
    "lora": {"r": 16, "alpha": 32, "dropout": 0.05, "target": "all-linear"},
    "sft": {"epochs": args.num_train_epochs, "effective_batch": 16,
            "lr": args.learning_rate, "max_length": args.max_length, "seed": args.seed},
    "gpu": torch.cuda.get_device_name(0),
    "smoke": SMOKE,
    "gguf": os.path.basename(GGUF_OUT),
    "date": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
with open(f"{ART}/manifest_8b_v1.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))

## 12 · Self-terminate
Removes (not just stops) the pod so billing ends; `/workspace` survives. Guarded so a
broken/missing GGUF never silently terminates the run.

In [ ]:
if ON_RUNPOD:
    assert os.path.exists(GGUF_OUT) and os.path.getsize(GGUF_OUT) > 5e9, \
        "final GGUF missing or suspiciously small — NOT terminating, investigate first"
    !ls -lh {ART}
    !sync
    # runpodctl ships on every pod with a pod-scoped key; remove = terminate + free the GPU
    !runpodctl remove pod $RUNPOD_POD_ID
else:
    print("not on RunPod — skipping self-terminate (smoke run complete)")

## Afterwards — pulling the GGUF to the Mac

The pod is gone; the artifacts live on the network volume. Cheapest retrieval:
1. Deploy the cheapest **CPU pod** in the same DC with the volume attached.
2. Either download `artifacts/qwen-coach-8b-v1-q4_k_m.gguf` through Jupyter, or run
   `runpodctl send /workspace/artifacts/qwen-coach-8b-v1-q4_k_m.gguf` on the pod and
   `runpodctl receive <code>` on the Mac (`brew install runpod/runpodctl/runpodctl`).
3. Delete the CPU pod (and, once the GGUF is safely local, consider deleting the volume —
   it bills ~$0.07/GB/month even when idle).

Then on the Mac: `ollama create qwen-coach-8b-v1 -f Modelfile` (same Modelfile as v1, new
GGUF path) → `python -m eval.run --coach-model qwen-coach-8b-v1 --judge anthropic` →
`python -m eval.compare` against the 29-game baseline.

**Heads-up:** an 8B q4_k_m is ~5 GB — comfortable on the 16 GB M1 Pro. Qwen3 defaults to
thinking mode in Ollama: before `eval.run`, update `coach/client.py` to disable thinking
(or strip `<think>` blocks) so the coach emits plain text — small Mac-side PR.